Setup

In [7]:
import shutil
import os
from langchain_community.document_loaders import DirectoryLoader, UnstructuredMarkdownLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings

In [8]:
DOCS_PATH = "./docs"  # Path to fraud rules documents
CHROMA_PATH = "./chroma_db"

In [9]:
# load markdown files
md_loader = DirectoryLoader(
    DOCS_PATH,
    glob="**/*.md",
    loader_cls=UnstructuredMarkdownLoader,
    show_progress=True
)
md_docs = md_loader.load()
print(f"Loaded {len(md_docs)} markdown files")

100%|██████████| 1/1 [00:00<00:00, 20.51it/s]

Loaded 1 markdown files


In [10]:
# Load text files
txt_loader = DirectoryLoader(
    DOCS_PATH,
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=True
)
txt_docs = txt_loader.load()
print(f"Loaded {len(txt_docs)} text files")

0it [00:00, ?it/s]

Loaded 0 text files


In [11]:
import re
from langchain_core.documents import Document

all_docs = md_docs + txt_docs
print(f"Total documents: {len(all_docs)}")

# Secondary splitter — used after rule-boundary split
secondary_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000,
    chunk_overlap=200
)

def split_by_rule_sections(doc):
    """Pre-split a markdown doc at rule headings before character chunking.
    Keeps each rule's header, query logic, and summary in the same chunk(s).
    """
    parts = re.split(r'(?=\n\d+\. # Rule-\d+:)', doc.page_content)
    return [
        Document(page_content=p, metadata=doc.metadata)
        for p in parts if p.strip()
    ]

chunks = []
for doc in md_docs:
    sections = split_by_rule_sections(doc)
    chunks.extend(secondary_splitter.split_documents(sections))

chunks.extend(secondary_splitter.split_documents(txt_docs))

print(f"Split into {len(chunks)} chunks")

Total documents: 1
Split into 11 chunks


In [12]:
#Embed & store

embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Wipe existing DB for a clean ingest
if os.path.exists(CHROMA_PATH):
    shutil.rmtree(CHROMA_PATH)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=CHROMA_PATH
)

In [13]:
print("✅ Done —", len(chunks), "chunks stored in chroma_db")

✅ Done — 11 chunks stored in chroma_db
